# Five Applied Machine Learning Techniques Implemented in Python

This portfolio project demonstrates five applied machine learning workflows:

1. Transformer-based sentiment analysis with Hugging Face
2. Evolutionary hyperparameter optimization
3. Interpretable decision-tree classification
4. Specialized and ensemble learning
5. Explainable AI with LIME

Each section includes data preparation, model development, evaluation, and a discussion of practical tradeoffs. The notebook is designed to run from a cloned copy of this GitHub repository using project-relative paths.

## Project Setup

The notebook uses project-relative paths rather than machine-specific file locations. Run it from either the repository root or the `notebooks/` directory.

The transformer section may require substantially more time on a CPU than on a CUDA-enabled GPU. Model files are downloaded from Hugging Face the first time that section is run.

In [ ]:
from pathlib import Path


def find_project_root(start_path: Path) -> Path:
    """
    Locate the repository root by searching the current directory
    and its parent directories for the data folder and requirements file.
    """
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the cloned repository root.\n\n"
        "Expected a parent directory containing:\n"
        "  - data/\n"
        "  - requirements.txt\n\n"
        f"Current Jupyter working directory:\n{start_path}\n\n"
        "Open and run the notebook from inside the cloned repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
IMAGES_DIR = PROJECT_ROOT / "images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {PROJECT_ROOT}")
print(f"Data directory:   {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Images directory: {IMAGES_DIR}")

# 1. Transformer-Based Sentiment Analysis

## 1.1 Load and Prepare the IMDB Movie Reviews Dataset


In [ ]:
import pandas as pd

imdb_path = DATA_DIR / "imdb" / "IMDB Dataset.csv"

if not imdb_path.is_file():
    raise FileNotFoundError(
        "The IMDB dataset was not found.\n\n"
        f"Expected location:\n{imdb_path}\n\n"
        "Download the IMDB Dataset of 50K Movie Reviews and place "
        "'IMDB Dataset.csv' in data/imdb/."
    )

df = pd.read_csv(imdb_path)

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0,
})
df = df.drop(columns=["sentiment"])

print(df.head())

## 1.2 Establish a Pretrained Transformer Baseline

Use the BERT model from Hugging Face to perform sentiment analysis.

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment")

The current pytorch version installed on my device does not have access to running tasks on GPU. As such, I have it running on my CPU which makes the runtimes incredibly slow. Since I have other work dependent on the version of pytorch installed, I decided not to mess around with getting GPU capability running for this programming assignment. 

In order to reduce the exceedingly long runtimes that BERT would take on the full 50000 row dataset, I decided to reduce the data by a factor of 10. The remaining 5000 rows are expected to make runtime in later scipts more reasonable. 2500 positive and 2500 negative - both randomly selected - rows to ensure validity and balance of the data.

In [ ]:
df_pos = df[df['label'] == 1].sample(n=2500, random_state=42)
df_neg = df[df['label'] == 0].sample(n=2500, random_state=42)
df_subset = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# 5000 reviews = ~ 13 min runtime

classification_results = classifier(
    df_subset['review'].tolist(),
    truncation=True,
    padding=True,
    max_length=512,
    batch_size=16
)

# to DataFrame
df_subset['predicted_label'] = [res['label'] for res in classification_results]
df_subset['predicted_score'] = [res['score'] for res in classification_results]
df_subset['predicted_stars'] = df_subset['predicted_label'].str.extract(r'(\d+)', expand=False).astype(int)
star_to_class = {1: 0, 2: 0, 3: -1, 4: 1, 5: 1}
df_subset['pred_label'] = df_subset['predicted_stars'].map(star_to_class).astype('int8')

df_subset = df_subset.drop(columns=['predicted_label'])

print(df_subset.head())

This basic classification looks at the reviews of a random subset of 5000 reviews in imdb dataset to determine if it is positive, negative, or neutral (since the original BERT model breaks it down by starts 1-5)

In [ ]:
num_neutral = (df_subset['pred_label'] == -1).sum()
print(f"Rows with pred_label = -1 (neutral): {num_neutral}")


Since with the model used for BERT does not have the ability to differential between positive and negative for reviews that are determined to be neutral, we will randomly assign half of these to positive, and half to negative. Given the balance of the original subset of data we got (50% positive and 50% negative), it is logical to assume the neutral outputs also fall along this distribution

In [ ]:
import numpy as np

# Indices of neutral rows
neutral_idx = df_subset.index[df_subset['pred_label'] == -1].to_numpy()
n = neutral_idx.size
if n > 0:
    # Reproducible shuffle (change/omit seed if you want different splits each run)
    np.random.seed(42)
    shuffled = np.random.permutation(neutral_idx)

    # Half to positive (extra goes to positive if odd), half to negative
    n_pos = (n + 1) // 2
    pos_idx = shuffled[:n_pos]
    neg_idx = shuffled[n_pos:]

    df_subset.loc[pos_idx, 'pred_label'] = 1
    df_subset.loc[neg_idx, 'pred_label'] = 0

print(f"Neutral reassigned -> +1: {len(pos_idx)}, 0: {len(neg_idx)}")


Now that we have assigned BERT to the data directly, we can move on to training the model.

## 1.3 Fine-tune the BERT model using DistilBERT on the IMDB dataset.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
import torch

# Split
df_trainable = df_subset[['review', 'label']].dropna().copy()
train_df, test_df = train_test_split(
    df_trainable, test_size=0.2, random_state=42, stratify=df_trainable['label']
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LEN = 128
def tok(b): return tokenizer(b["review"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tok, batched=True, remove_columns=train_ds.column_names)
test_tok  = test_ds.map(tok,  batched=True, remove_columns=test_ds.column_names)
train_tok = train_tok.add_column("labels", train_df["label"].astype(int).tolist())
test_tok  = test_tok.add_column("labels",  test_df["label"].astype(int).tolist())

collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# MINIMAL TrainingArguments
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "distilbert_imdb_ft"),
    num_train_epochs=1,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 8,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=collator
)

trainer.train()



## 1.4 Qualitative Predictions on New Text

In [ ]:
# Use the trained `model` and `tokenizer` to predict on new texts
import torch

texts = [
    "I love studying Natural Language Processing.",
    "I am so tired today.",
    "The movie was fantastic!",
    "The service was terrible."
]

# Encode
enc = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128,     # match your training max_length
    return_tensors="pt"
)

device = next(model.parameters()).device
enc = {k: v.to(device) for k, v in enc.items()}

model.eval()
with torch.no_grad():
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=1)

# Map 0/1 to human-friendly labels
id2name = {0: "negative", 1: "positive"}

for text, pid, p in zip(texts, pred_ids, probs):
    print(f"{id2name[int(pid)]:>8}  score={p[pid]:.3f}  |  {text}")


## 1.5 Evaluate the Fine-Tuned Model


In [ ]:
# Evaluate on the held-out test set with accuracy, precision, recall, F1
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

pred = trainer.predict(test_tok)
y_true = pred.label_ids
y_pred = pred.predictions.argmax(axis=1)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)

print({
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
})


## 1.6 Inspect Sample Predictions


In [ ]:
import numpy as np
import pandas as pd
import torch

SAMPLES = 10
sample_df = df_subset.sample(n=SAMPLES, random_state=42).copy()

# predict with fine-tuned model
def predict_labels(texts, model, tokenizer, max_len=128):
    enc = tokenizer(
        list(texts),
        truncation=True, padding=True, max_length=max_len,
        return_tensors="pt"
    )
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=1)
    return preds, probs

ft_pred_ids, ft_probs = predict_labels(sample_df["review"], model, tokenizer, max_len=128)

# build a small comparison table
id2name = {0: "negative", 1: "positive"}
tri_id2name = {-1: "neutral", 0: "negative", 1: "positive"}

out = pd.DataFrame({
    "review_excerpt": sample_df["review"].str.replace(r"\s+", " ", regex=True).str.slice(0, 180) + "...",
    "true_label": sample_df["label"].map(id2name),
    "five_star_pred": sample_df["pred_label"].map(tri_id2name),  # from your earlier 5-star mapping
    "ft_pred": [id2name[i] for i in ft_pred_ids],
    "ft_score": [float(ft_probs[i, ft_pred_ids[i]]) for i in range(len(sample_df))],
})

#  flag correctness
out["ft_correct"] = (out["true_label"] == out["ft_pred"])

print(out.to_string(index=False))


## 1.7 Results and Discussion


i) The held out test split shows accuracy around 0.848 with precision about 0.849, recall about 0.846, and an F1 score near 0.848. Precision being a touch higher than recall suggests the model is a bit conservative about calling something positive and avoids false positives slightly more than it misses true positives. Given the balanced subset, this level of performance is solid for a fast fine tune. The metrics are also very consistent with each other, which points to a well calibrated classifier for this setup.

ii) Because we trained and evaluated on a balanced mix of positive and negative reviews, performance is fairly even across both classes. The small edge of precision over recall for the positive class hints that the model is slightly better at confirming a review is positive than it is at catching every positive example. In practice that means fewer glowing ratings are mislabeled as negative, at the cost of missing a few borderline positives. Errors most likely come from mixed tone or neutral phrasing where sentiment is subtle.

iii) The biggest challenge early on was the mismatch between the five star pipeline outputs and the binary ground truth, which initially led to incorrect label handling. Runtime was another pain point, especially with long sequence lengths and a heavier model. We resolved this by using a lighter encoder, shortening the max sequence length, and simplifying training to one epoch, which kept total time in the requested window. Cleaning up the label mapping and avoiding confidence thresholds for class decisions also removed a common source of error.

iv) On the sample texts the model behaved as you would expect. Enthusiastic sentences like “I love studying Natural Language Processing” and “The movie was fantastic” were predicted as positive with strong confidence. A clear complaint like “The service was terrible” landed as negative with high confidence. The tired statement reads as slightly negative in sentiment even though it is not a movie opinion, which shows the model is sensitive to affect but can still conflate mood and review sentiment when context is thin.

v) If you want a step up from here, consider a small number of additional epochs with early stopping and keep max length modest to control time. Training a three class variant that explicitly models neutral could reduce confusion on balanced or lukewarm reviews. You could also try a stronger small model and compare results, then choose the best trade off between accuracy and speed. Finally, add a quick error analysis on the most confident mistakes to guide targeted data cleaning or lightweight augmentation.

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 2. Evolutionary Hyperparameter Optimization


## 2.1 Load the Red Wine Quality Dataset


In [ ]:
import pandas as pd

wine_path = DATA_DIR / "wine_quality" / "winequality-red.csv"

if not wine_path.is_file():
    raise FileNotFoundError(
        "The wine-quality dataset was not found.\n\n"
        f"Expected location:\n{wine_path}"
    )

df = pd.read_csv(wine_path)

df.head()

## 2.2 Optimize a Random Forest with Evolution Strategies


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = df.drop('quality', axis=1)
y = df['quality']

# define  fitness function
def fitness_function(params):
    n_estimators = int(params[0])
    max_depth = int(params[1])

    model = make_pipeline(
        StandardScaler(),
        RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    )
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    return -np.mean(scores)

# ES algorithm - for sake of time, only 5 evolutions
def evolution_strategy(fitness_fn, bounds, population_size=30, sigma=1.0, alpha=0.1, iterations=10):
    dim = len(bounds)
    mean = np.array([(low + high) / 2 for (low, high) in bounds])
    history = []

    for i in range(iterations):
        population = [mean + sigma * np.random.randn(dim) for _ in range(population_size)]
        population = np.clip(population, [b[0] for b in bounds], [b[1] for b in bounds])
        scores = [fitness_fn(ind) for ind in population]

        best_idx = np.argmin(scores)
        best = population[best_idx]

        mean = mean + alpha * (best - mean)
        history.append((i, scores[best_idx]))
        print(f"Iteration {i+1}: Best MSE = {scores[best_idx]:.4f} with Params = {best}")

    return mean, history

# Define bounds for RandomForest
param_bounds = [(10, 200), (2, 20)]


best_params, history = evolution_strategy(fitness_function, param_bounds)
best_n_estimators, best_max_depth = int(best_params[0]), int(best_params[1])

print(f"\nBest Params from ES: n_estimators={best_n_estimators}, max_depth={best_max_depth}")


## 2.3 Compare Evolutionary Search with Grid Search


In [ ]:
from sklearn.model_selection import GridSearchCV

# Grid search
pipeline = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(random_state=42)
)

param_grid = {
    'randomforestregressor__n_estimators': [50, 100, 150],
    'randomforestregressor__max_depth': [5, 10, 15]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X, y)



print("Grid Search Best Params:", grid_search.best_params_)
print(f"Grid Search Best MSE: {-grid_search.best_score_:.4f}")

## 2.4 Analyze Convergence and Solution Quality


In [ ]:
import matplotlib.pyplot as plt

# Plot convergence curve of ES
iterations, mse_scores = zip(*history)

plt.figure(figsize=(10, 6))
plt.plot(iterations, mse_scores, marker='o', label='Evolution Strategies')
plt.axhline(-grid_search.best_score_, color='r', linestyle='--', label='Grid Search Best')
plt.xlabel('Iteration')
plt.ylabel('Best MSE Found')
plt.title('Convergence of Evolution Strategies vs Grid Search')
plt.legend()
plt.grid(True)
plt.show()

# Final comparison
print(f"\nFinal Comparison:")
print(f"Evolution Strategies Best MSE: {min(mse_scores):.4f}")
print(f"Grid Search Best MSE: {-grid_search.best_score_:.4f}")


## 2.5 Results and Discussion


i) For this regression task, we used Mean Squared Error (MSE) as the main evaluation metric. MSE calculates the average squared difference between the predicted and actual values, where smaller values indicate more accurate predictions. It’s widely used in regression because it penalizes large errors more heavily than small ones. Since we are not dealing with classification, metrics like accuracy, precision, recall, and F1-score were not applicable. To get more reliable results, we used five-fold cross-validation during model evaluation.

ii) In this run, the Evolution Strategies method found a best MSE of 0.4176, slightly outperforming Grid Search’s best MSE of 0.4177. While the improvement was small, it shows that ES was able to continue refining its solution across more iterations. The convergence curve confirms this slow but steady improvement, as ES moved from initial fluctuations to a better result than Grid Search. This suggests that with enough iterations and proper configuration, ES can be an effective optimization method for tuning model hyperparameters. The fact that ES was not handcuffed to a fixed grid also gave it a bit more flexibility in the search.

iii) One of the early issues with Evolution Strategies was keeping parameter values within valid ranges. This was addressed using a clipping step that made sure each candidate stayed inside the acceptable bounds. Another challenge was the variability in model performance caused by different data splits, which was managed by averaging results over five cross-validation folds. Tracking each iteration's result helped us understand how the search progressed and ensured that things were moving in the right direction. While the process worked well, future versions could benefit from tuning ES-specific settings to improve efficiency and consistency. Another issue was the computation time of the ES, which was about one minute for every evolution. I couldn't find a more efficient way to do ES, so I decided to reduce the number of evolutions to just 10. More evolutions would lead to even better results possibly.

iv) The updated convergence plot shows that Evolution Strategies started with some fluctuation but then improved steadily over time. While the initial values bounced around, the last few iterations showed a clear trend toward lower MSE. Eventually, ES reached a score just below that of Grid Search, indicating successful convergence. The fact that ES kept improving while staying close to the best solution suggests that the search space still had valuable areas to explore. This behavior confirms that running ES for more iterations was worthwhile.

v) In this experiment, Evolution Strategies came out just ahead of Grid Search, though the difference was very small. Grid Search is simple and effective when you know what parameter values to try, but it can miss better combinations that fall between the fixed choices. ES, by contrast, searches continuously and does not depend on predefined grids. This makes it more flexible and better suited for larger or less predictable search spaces. The fact that ES performed slightly better here highlights its potential value, especially when more complex models are involved.

vi) To take this work further, it would help to run Evolution Strategies for even more iterations and experiment with a wider range of hyperparameters. Adding more variables such as minimum samples per split or maximum number of features could open up better combinations. Another interesting direction would be to frame the problem as a classification task by grouping wine scores into categories and using metrics like accuracy and F1 score. It may also be helpful to try different types of evolutionary methods like Genetic Algorithms or Bayesian optimization. Finally, combining the strengths of both approaches by starting with a good result from Grid Search and letting ES continue from there might give the best overall performance.


---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 3. Interpretable Decision-Tree Classification


## 3.1 Load the Titanic Survival Dataset


In [ ]:
import pandas as pd

titanic_path = DATA_DIR / "titanic" / "titanic.csv"

if not titanic_path.is_file():
    raise FileNotFoundError(
        "The Titanic dataset was not found.\n\n"
        f"Expected location:\n{titanic_path}"
    )

df = pd.read_csv(titanic_path)

print(df.head())

## 3.2 Prepare the Data and Train a Baseline Decision Tree


In [ ]:
missing_summary = df.isnull().sum()
print(missing_summary)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Preprocessing function
def preprocess(df):
    df = df.copy()
    
    df = df.drop(columns=['Cabin', 'Fare', 'Ticket', 'PassengerId', 'Name'])  # Note: 'PassengerId' must match exact spelling

    # Fill missing values
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna('S')

    # Encode categorical variables
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

#    df.to_csv(
#        OUTPUT_DIR / "titanic_preprocessed.csv",
#        index=False,
#    )

    X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Embarked']]
    y = df['Survived']

    return X, y

X, y = preprocess(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)


## 3.3 Tune Hyperparameters with Cross-Validation


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

X, y = preprocess(df) 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

# grid search with 5-fold cross-validation on training set only
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

# model and evaluate on the test set
best_model = grid_search.best_estimator_
print("Best Parameters from Grid Search:", grid_search.best_params_)


## 3.4 Evaluate Classification Performance


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred = best_model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Full classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))


## 3.5 Visualize the Tree and Feature Importance


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
import pandas as pd

plt.figure(figsize=(24, 12), dpi=100)

plot_tree(
    best_model,
    feature_names=X.columns,
    class_names=['Not Survived', 'Survived'],
    filled=True,
    rounded=True,
    fontsize=20
)

plt.title("Decision Tree - Titanic Survival", fontsize=40)
plt.tight_layout()
plt.show()


## 3.6 Results and Discussion


i) The test metrics came out as accuracy 1.00, precision 1.00, recall 1.00, and F1 score 1.00, which means every test case was predicted correctly. Because perfect scores are rare, it strongly hints at possible data leakage or an unusually easy split. I tried to prevent leakage by dropping identifiers like PassengerId and Ticket, keeping encodings clean, imputing sensibly, and running the grid search only on the training folds, but I could not uncover any leaking path. Maybe this is the slim chance where the model really is perfect on this test set, though I would still sanity check with new random seeds and cross validation.

ii) Feature importance is completely dominated by Sex, which makes sense based on "women and children first" mentality of the titanic. The learned tree makes a single decision at the root on Sex and never touches Pclass, Age, SibSp, Parch, or Embarked. You can see the impurity drop from a gini of about 0.457 at the root to pure leaves, which means the Sex split alone cleans up all the uncertainty on this split of data. In short, the model behaves like a simple rule based on gender.

iii) The main challenge was getting a clean modeling table and a search space that does not overcomplicate things. Handling missing Age and Embarked values and keeping encodings consistent was necessary to avoid crashes and silent errors. The grid search was modest and ended up selecting a shallow tree with the gini criterion, which kept the model simple and fast. Once the preprocessing was stable, the rest was straightforward. Obviously preventing leakage was also a challenge that I wasn't quite able to solve.

iv) The visualization shows a single split on Sex with males going left and being labeled as not survived and females going right and being labeled as survived. Both leaves have gini equal to zero, meaning each leaf contains only one class on this data. The counts in the boxes line up with that story and there are no further branches because the first split already produced pure nodes. It is the simplest possible tree and it mirrors the well known pattern from the Titanic story.

v) The perfect scores suggest this train and test split was unusually separable, so I would first try a stratified split on Survived to ensure a more representative test set and then repeat cross validation. I would also add checks for leakage and try different random seeds to see if performance holds. As next steps, compare against a logistic regression and a random forest, report ROC AUC and a confusion matrix, and examine permutation importance to verify that Sex is not the only useful signal. If you want richer trees, consider limiting class purity with min samples per leaf or adding class weights so the model explores additional features like Pclass and Age.

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 4. Specialized and Ensemble Learning for Evergreen Web Classification


## 4.1 Load the StumbleUpon Evergreen Dataset


In [ ]:
import pandas as pd

evergreen_train_path = DATA_DIR / "evergreen" / "evergreen_train.tsv"
evergreen_test_path = DATA_DIR / "evergreen" / "evergreen_test.tsv"

missing_files = [
    path
    for path in [evergreen_train_path, evergreen_test_path]
    if not path.is_file()
]

if missing_files:
    missing_text = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(
        "One or more Evergreen dataset files were not found:\n"
        f"{missing_text}"
    )

train_df = pd.read_csv(
    evergreen_train_path,
    sep="\t",
    header=0,
)

test_df = pd.read_csv(
    evergreen_test_path,
    sep="\t",
    header=0,
)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

train_df.head()

train_df.to_csv(
    OUTPUT_DIR / "train_evergreen_df.csv",
    index=False,
)

## 4.2 Preprocess Structured and Categorical Features

### 4.2.1 Identify and Impute Missing Values


The dataset represents some missing values with a question mark rather than a standard null value. These entries are identified before imputation.


In [ ]:
# Count '?' in each column for train
print("Count of '?' in training set:")
print((train_df == '?').sum())

# Count '?' in each column for test
print("\nCount of '?' in test set:")
print((test_df == '?').sum())


In [ ]:
import numpy as np

train_imp = train_df.copy()
test_imp  = test_df.copy()

cols_with_q = ['alchemy_category_score', 'is_news', 'news_front_page']

# '?' -> NaN
train_imp[cols_with_q] = train_imp[cols_with_q].replace('?', np.nan)
test_imp[cols_with_q]  = test_imp[cols_with_q].replace('?', np.nan)


for col in cols_with_q:
    train_imp.loc[:, col] = pd.to_numeric(train_imp[col], errors='coerce')
    test_imp.loc[:, col]  = pd.to_numeric(test_imp[col],  errors='coerce')


median_score = train_imp['alchemy_category_score'].median()
mode_is_news = train_imp['is_news'].mode(dropna=True)[0]
mode_front   = train_imp['news_front_page'].mode(dropna=True)[0]


fill_map = {
    'alchemy_category_score': median_score,
    'is_news': mode_is_news,
    'news_front_page': mode_front
}
train_imp[cols_with_q] = train_imp[cols_with_q].fillna(fill_map)
test_imp[cols_with_q]  = test_imp[cols_with_q].fillna(fill_map)


train_imp.to_csv(
    OUTPUT_DIR / "evergreen_train_imputed.csv",
    index=False,
)



print("Remaining missing values in train:\n",
      train_imp[cols_with_q].isna().sum())
print("\nRemaining missing values in test:\n",
      test_imp[cols_with_q].isna().sum())


For the numeric field alchemy_category_score, we replaced missing values with the median from the training data to preserve central tendency without being skewed by outliers.
For the binary categorical fields is_news and news_front_page, we replaced missing values with the most frequent value (mode) from the training data to maintain the majority class distribution.

### 4.2.2 Encode Categorical Variables


In [ ]:
from pandas.api.types import CategoricalDtype
from urllib.parse import urlparse

train_cat = train_imp.copy()
test_cat  = test_imp.copy()

train_cat['url_domain'] = train_cat['url'].apply(lambda x: urlparse(x).netloc)
test_cat['url_domain']  = test_cat['url'].apply(lambda x: urlparse(x).netloc)

# build vocab from train
dom_dtype = CategoricalDtype(categories=train_cat['url_domain'].unique())
cat_dtype = CategoricalDtype(categories=train_cat['alchemy_category'].unique())

# cast test to those categories; unknowns become NaN, whose .cat.codes == -1
train_cat['url_domain_enc'] = train_cat['url_domain'].astype(dom_dtype).cat.codes
test_cat['url_domain_enc']  = test_cat['url_domain'].astype(dom_dtype).cat.codes

train_cat['alchemy_category_enc'] = train_cat['alchemy_category'].astype(cat_dtype).cat.codes
test_cat['alchemy_category_enc']  = test_cat['alchemy_category'].astype(cat_dtype).cat.codes

# Remove categorical string columns
train_cat = train_cat.drop(columns=['url', 'url_domain', 'alchemy_category'])

train_cat.to_csv(
    OUTPUT_DIR / "evergreen_train_encoded.csv",
    index=False,
)

# Preview result
train_cat.head()


Since each url is unique and has a corresponding numerical urlid, I determined the domain of each url and encoded based on that - creating url_domain and url_domain_enc. I then removed the original url column and url_domain.
I encoded the only reamaining categorical field - alchemy_category. I am keeping the page word content (boilerplate) field for now.

### 4.2.3 Standardize Numeric Features


In [ ]:
from sklearn.preprocessing import StandardScaler

train_scaled = train_cat.copy()
test_scaled  = test_cat.copy()

# Explicit boolean/indicator columns to exclude from scaling
bool_cols = [
    'is_news', 'framebased', 'hasDomainLink',
    'lengthyLinkDomain', 'news_front_page'
]

# Select numeric columns but exclude label, urlid, and boolean fields
num_cols = train_scaled.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_cols = [col for col in num_cols if col not in ['label', 'urlid'] + bool_cols]

scaler = StandardScaler()

# Transform both train & test for selected numeric columns
train_scaled[num_cols] = scaler.fit_transform(train_imp[num_cols])
test_scaled[num_cols]  = scaler.transform(test_imp[num_cols])

# Keep urlid unscaled
train_scaled['urlid'] = train_imp['urlid']
test_scaled['urlid']  = test_imp['urlid']

# Keep boolean/indicator fields as original
for col in bool_cols:
    train_scaled[col] = train_imp[col]
    test_scaled[col]  = test_imp[col]

scaled_output_path = OUTPUT_DIR / "evergreen_train_scaled.csv"

# train_scaled.to_csv(
#     scaled_output_path,
#     index=False,
# )

# Check scaling stats for scaled cols only
print(train_scaled[num_cols].mean().round(3))
print(train_scaled[num_cols].std().round(3))


This code scaled all numeric features (except the label) using the training set’s mean and standard deviation, then applied the same transformation to the test set to avoid leakage.
Now our data is ready for algorithms sensitive to feature scale, improving stability, speed, and performance.

Boolean fields were excluded in addition to the urlid.

### 4.2.4 Outlier Considerations


Potential outliers are retained because several models used later are tree-based or regularized and can tolerate moderate distributional extremes. Standardization reduces scale differences for linear and distance-based models. A production workflow could add robust scaling or winsorization after validating the effect through cross-validation.


## 4.3 Extract Text and Metadata Features


### 4.3.1 TF-IDF Features from Page Content


In [ ]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

train_feat = train_scaled.copy()
test_feat  = test_scaled.copy()


def extract_text(x):
    if pd.isna(x): return ""
    try:
        bp = json.loads(x)
        return f"{bp.get('title','')} {bp.get('body','')}".strip()
    except Exception:
        return str(x)

train_text = train_feat["boilerplate"].apply(extract_text)
test_text  = test_feat["boilerplate"].apply(extract_text)

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=5, max_features=50000)
X_train_text_tfidf = tfidf.fit_transform(train_text) 
X_test_text_tfidf  = tfidf.transform(test_text)



# store boilerplate as feature values
X_train_num = train_feat.drop(columns=["boilerplate","label"]).select_dtypes(include=[np.number]).values
X_test_num  = test_feat.drop(columns=["boilerplate"]).select_dtypes(include=[np.number]).values

X_train_tfidf_plus = hstack([X_train_text_tfidf, X_train_num])
X_test_tfidf_plus  = hstack([X_test_text_tfidf,  X_test_num])


In [ ]:
# sizes
print("X_train_tfidf_plus:", X_train_tfidf_plus.shape)
print("X_test_tfidf_plus: ", X_test_tfidf_plus.shape)

num_cols_used = train_feat.drop(columns=["boilerplate","label"]).select_dtypes(include=[np.number]).columns.tolist()
print("TF-IDF features:", len(tfidf.get_feature_names_out()))
print("Numeric features:", len(num_cols_used))
print("First 10 TF-IDF features:", tfidf.get_feature_names_out()[3000:3010]) 
print("First 10 numeric columns:", num_cols_used[:10])


Our matrix above identifies the available features (words) in the boilerplate field

### 4.3.2 Metadata Features


The dataset does not provide direct social-engagement fields such as shares, likes, or comments. The structured metadata available in the dataset is therefore used as the non-text feature set.


The original dataset does not include explicit engagement measurements. Rather than infer unavailable variables, the analysis uses observed metadata fields and documents this limitation.


### 4.3.3 Feature-Label Relationships


In [ ]:
# Labels
y = train_feat['label'].astype(int).values

# Convert to dense (careful with RAM)
X_dense = X_train_text_tfidf.toarray()

# correlations for each word with the label
correlations = [np.corrcoef(X_dense[:, i], y)[0, 1] for i in range(X_dense.shape[1])]

corr_df = pd.DataFrame({
    'feature': tfidf.get_feature_names_out(),
    'correlation': correlations
})

top_pos = corr_df.sort_values('correlation', ascending=False).head(20)
top_neg = corr_df.sort_values('correlation', ascending=True).head(20)

print("\nTop positive correlated terms with label=1:")
print(top_pos)

print("\nTop negative correlated terms with label=1:")
print(top_neg)

The above feature importance shows the features extracted from boilerplate that are most likely to positively or negatively influence whether a website is evergreen or not.

In [ ]:
# Get numeric metadata correlations with label
metadata_cols = [
    'alchemy_category_score', 'avglinksize', 'commonlinkratio_1', 'commonlinkratio_2',
    'commonlinkratio_3', 'commonlinkratio_4', 'compression_ratio', 'embed_ratio',
    'framebased', 'frameTagRatio', 'hasDomainLink', 'html_ratio', 'image_ratio',
    'is_news', 'lengthyLinkDomain', 'linkwordscore', 'news_front_page',
    'non_markup_alphanum_characters', 'numberOfLinks', 'numwords_in_url',
    'parametrizedLinkRatio', 'spelling_errors_ratio', 'url_domain_enc',
    'alchemy_category_enc'
]

meta_corrs = []
for col in metadata_cols:
    corr = np.corrcoef(train_feat[col], y)[0, 1]
    meta_corrs.append((col, corr))

meta_corr_df = pd.DataFrame(meta_corrs, columns=['feature', 'correlation']).sort_values('correlation', ascending=False)

print("\nTop positive correlated metadata features with label=1:")
print(meta_corr_df.head(10))

print("\nTop negative correlated metadata features with label=1:")
print(meta_corr_df.tail(10))


The above feature importances show the correlation of metadata features to positively or negatively influence whether a website is evergreen or not.

## 4.4 Segment Content for Specialized Modeling

### 4.4.1 Define Text-Heavy and Metadata-Heavy Segments


Rows are segmented by comparing normalized text intensity with the mean absolute magnitude of scaled metadata features. This heuristic provides a transparent routing rule for the specialist models.


In [ ]:
metadata_cols = [
    'alchemy_category_score','avglinksize','commonlinkratio_1','commonlinkratio_2',
    'commonlinkratio_3','commonlinkratio_4','compression_ratio','embed_ratio',
    'framebased','frameTagRatio','hasDomainLink','html_ratio','image_ratio','is_news',
    'lengthyLinkDomain','linkwordscore','news_front_page',
    'non_markup_alphanum_characters','numberOfLinks','numwords_in_url',
    'parametrizedLinkRatio','spelling_errors_ratio','url_domain_enc','alchemy_category_enc'
]

# L2 norm of TF-IDF row
train_text_intensity = np.sqrt(X_train_text_tfidf.power(2).sum(axis=1)).A1
test_text_intensity  = np.sqrt(X_test_text_tfidf.power(2).sum(axis=1)).A1

# mean absolute value of already-scaled metadata
train_meta_intensity = train_feat[metadata_cols].astype(float).abs().mean(axis=1).to_numpy()
test_meta_intensity  = test_feat[metadata_cols].astype(float).abs().mean(axis=1).to_numpy()

# whichever intensity is greater wins
def assign_segments(text_int, meta_int):
    seg = np.where(text_int > meta_int, 'text-heavy',
                   np.where(meta_int > text_int, 'metadata-heavy', 'mixed'))
    return seg

train_feat['segment'] = assign_segments(train_text_intensity, train_meta_intensity)
test_feat['segment']  = assign_segments(test_text_intensity,  test_meta_intensity)

print(train_feat['segment'].value_counts())


### 4.4.2 Train Segment-Specific Models


In [ ]:
from sklearn.linear_model import LogisticRegression

# Train both models on the full train set 
text_clf = LogisticRegression(max_iter=1000, class_weight='balanced')
meta_clf = LogisticRegression(max_iter=1000, class_weight='balanced')

text_clf.fit(X_train_text_tfidf, y)
meta_clf.fit(X_train_num, y)

# route test rows by segment
seg_te = test_feat['segment'].to_numpy()

mask_text  = (seg_te == 'text-heavy')
mask_meta  = (seg_te == 'metadata-heavy')
mask_mixed = ~(mask_text | mask_meta)   # ties or anything else → blend

proba = np.zeros(test_feat.shape[0], dtype=float)

# if text-heavy then text model
if mask_text.any():
    proba[mask_text] = text_clf.predict_proba(X_test_text_tfidf[mask_text])[:, 1]

# if metadata-heavy then metadata model
if mask_meta.any():
    proba[mask_meta] = meta_clf.predict_proba(X_test_num[mask_meta])[:, 1]

# if mixed then average of both specialists
if mask_mixed.any():
    p_text = text_clf.predict_proba(X_test_text_tfidf[mask_mixed])[:, 1]
    p_meta = meta_clf.predict_proba(X_test_num[mask_mixed])[:, 1]
    proba[mask_mixed] = 0.5 * p_text + 0.5 * p_meta

pred = (proba >= 0.5).astype(int)

submission = test_feat[['urlid']].copy()
submission['label_proba'] = proba
submission['label'] = pred
print(submission.head())

# submission.to_csv(
#     OUTPUT_DIR / "submission_two_specialists.csv",
#     index=False,
# )

## 4.5 Train and Tune Specialized Models


### 4.5.1 Text Specialist: Logistic Regression


In [ ]:
# Logistic Regression on text

# class-imbalance weighting
pos_w = (y == 0).sum() / max((y == 1).sum(), 1)
sample_w = np.where(y == 1, pos_w, 1.0)

logreg_text = LogisticRegression(max_iter=1000, class_weight=None)  # using sample_weight below
logreg_text.fit(X_train_text_tfidf, y, sample_weight=sample_w)

text_proba_test = logreg_text.predict_proba(X_test_text_tfidf)[:, 1]
text_pred_test  = (text_proba_test >= 0.5).astype(int)


In [ ]:
# Gradient Boosting on metadata

from sklearn.ensemble import GradientBoostingClassifier

gb_meta = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=42
)
gb_meta.fit(X_train_num, y, sample_weight=sample_w)

meta_proba_test = gb_meta.predict_proba(X_test_num)[:, 1]
meta_pred_test  = (meta_proba_test >= 0.5).astype(int)


### 4.5.2 Hyperparameter Tuning


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_grid = {
    "solver": ["liblinear", "saga"], 
    "penalty": ["l1", "l2"],
    "C": [0.05, 0.1, 0.5, 1.0, 2.0, 5.0],
}
lr = LogisticRegression(max_iter=3000, class_weight="balanced", n_jobs=None)

lr_cv = GridSearchCV(
    estimator=lr,
    param_grid=lr_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=0
)
lr_cv.fit(X_train_text_tfidf, y)

best_lr = lr_cv.best_estimator_
print("Best LR params:", lr_cv.best_params_)
print("Best LR CV AUC:", lr_cv.best_score_)

# test-time predictions
text_proba_test = best_lr.predict_proba(X_test_text_tfidf)[:, 1]
text_pred_test  = (text_proba_test >= 0.5).astype(int)


In [ ]:

# long run time

gb = GradientBoostingClassifier(random_state=42)

gb_grid = {
    "n_estimators": [150, 300, 500],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.7, 0.85, 1.0],
    "max_features": [None, "sqrt", 0.5],
}

gb_cv = GridSearchCV(
    estimator=gb,
    param_grid=gb_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=0
)
gb_cv.fit(X_train_num, y)

best_gb = gb_cv.best_estimator_
print("Best GB params:", gb_cv.best_params_)
print("Best GB CV AUC:", gb_cv.best_score_)

meta_proba_test = best_gb.predict_proba(X_test_num)[:, 1]
meta_pred_test  = (meta_proba_test >= 0.5).astype(int)


## 4.6 Train Redundant Models and Build an Ensemble


### 4.6.1 Redundant Text Models


In [ ]:
# text models

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

# RF on TF-IDF
rf_text = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)
rf_text.fit(X_train_text_tfidf, y)
text_rf_pred = rf_text.predict(X_test_text_tfidf)


# SVM on TF-IDF 
svm_text_base = LinearSVC(random_state=42, class_weight="balanced")
# Calibrate to get probabilities
svm_text = CalibratedClassifierCV(svm_text_base, cv=5, method="sigmoid")
svm_text.fit(X_train_text_tfidf, y)
text_svm_pred = svm_text.predict(X_test_text_tfidf)


# text_svm_proba = svm_text.predict_proba(X_test_text_tfidf)[:, 1]


In [ ]:
# metadata models

# RF on metadata
rf_meta = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)
rf_meta.fit(X_train_num, y)
meta_rf_pred = rf_meta.predict(X_test_num)
# meta_rf_proba = rf_meta.predict_proba(X_test_num)[:, 1]

# RBF SVM on metadata 
svm_meta = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=42)
svm_meta.fit(X_train_num, y)
meta_svm_pred  = svm_meta.predict(X_test_num)


# meta_svm_proba = svm_meta.predict_proba(X_test_num)[:, 1]


### 4.6.2 Majority-Vote Ensemble


In [ ]:
# Majority Votinf

# binary predictions from each model
pred_text_lr  = (best_lr.predict_proba(X_test_text_tfidf)[:, 1] >= 0.5).astype(int)
pred_meta_gb  = (best_gb.predict_proba(X_test_num)[:, 1] >= 0.5).astype(int)
pred_text_rf  = rf_text.predict(X_test_text_tfidf)
pred_meta_rf  = rf_meta.predict(X_test_num)
pred_text_svm = svm_text.predict(X_test_text_tfidf)
pred_meta_svm = svm_meta.predict(X_test_num)

# Stack predictions
pred_matrix = np.column_stack([
    pred_text_lr,
    pred_meta_gb,
    pred_text_rf,
    pred_meta_rf,
    pred_text_svm,
    pred_meta_svm
])

# Majority vote: label = 1 if >= half of models vote 1
n_models = pred_matrix.shape[1]
maj_vote = (pred_matrix.sum(axis=1) >= (n_models // 2 + 1)).astype(int)

# Output example
submission = test_feat[['urlid']].copy()
submission['label'] = maj_vote
print(submission.head())

# submission.to_csv(
#     OUTPUT_DIR / "submission_majority_vote.csv",
#     index=False,
# )

## 4.7 Evaluate Performance and Robustness


### 4.7.1 Validation Metrics


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def eval_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"{name:<16}  ACC={acc:.3f}  PREC={prec:.3f}  REC={rec:.3f}  F1={f1:.3f}")

# Validation split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
(tr_idx, val_idx), = sss.split(X_train_text_tfidf, y)

Xtr_text, Xval_text = X_train_text_tfidf[tr_idx], X_train_text_tfidf[val_idx]
Xtr_num,  Xval_num  = X_train_num[tr_idx],         X_train_num[val_idx]
y_tr, y_val = y[tr_idx], y[val_idx]

# refit models on training fold
best_lr.fit(Xtr_text, y_tr)
best_gb.fit(Xtr_num,  y_tr)
rf_text.fit(Xtr_text, y_tr)
svm_text.fit(Xtr_text, y_tr)
rf_meta.fit(Xtr_num,  y_tr)
svm_meta.fit(Xtr_num,  y_tr)

# Predictions on validation fold
pred_text_lr  = (best_lr.predict_proba(Xval_text)[:,1] >= 0.5).astype(int)
pred_meta_gb  = (best_gb.predict_proba(Xval_num)[:,1]  >= 0.5).astype(int)
pred_text_rf  = rf_text.predict(Xval_text)
pred_meta_rf  = rf_meta.predict(Xval_num)
pred_text_svm = svm_text.predict(Xval_text)
pred_meta_svm = svm_meta.predict(Xval_num)

# Majority vote of all six models
pred_matrix = np.column_stack([
    pred_text_lr, pred_meta_gb, pred_text_rf, pred_meta_rf, pred_text_svm, pred_meta_svm
])
maj_vote = (pred_matrix.sum(axis=1) >= (pred_matrix.shape[1] // 2 + 1)).astype(int)

# Evaluate specialized models
print("Specialized Models:")
eval_model("Text LR",   y_val, pred_text_lr)
eval_model("Meta GB",   y_val, pred_meta_gb)

# Evaluate redundant models
print("\nRedundant Models:")
eval_model("Text RF",   y_val, pred_text_rf)
eval_model("Meta RF",   y_val, pred_meta_rf)
eval_model("Text SVM",  y_val, pred_text_svm)
eval_model("Meta SVM",  y_val, pred_meta_svm)

# Evaluate ensemble
print("\nCombined Ensemble:")
eval_model("Majority Vote", y_val, maj_vote)


### 4.7.2 Model Comparison


The specialized models clearly led the way, with the text-based Logistic Regression achieving the highest overall accuracy at 0.796 and the strongest precision at 0.872, along with a solid F1 of 0.781. The metadata-based Gradient Boosting model trailed in performance with 0.693 accuracy and a balanced precision and recall around 0.70, but still contributed useful structured-data insights. Among the redundant models, the text Random Forest (0.789 accuracy, 0.842 precision, 0.779 F1) and text SVM (0.784 accuracy, 0.823 precision, 0.777 F1) performed comparably to the text specialist, showing that alternative algorithms on the same feature space can still be competitive. The metadata Random Forest was similar to the metadata specialist, while the metadata SVM struggled with only 0.515 accuracy despite a relatively high recall of 0.747, indicating it overpredicted the positive class. When combined in a majority vote, the ensemble delivered a stable 0.784 accuracy, 0.853 precision, and 0.768 F1. This was slightly lower than the best specialist in raw accuracy, but the ensemble offered more balanced predictions and reduced reliance on a single model’s strengths. Overall, the specialists carried most of the predictive power, while the redundant models added diversity that occasionally helped correct specialist errors and provided a more consistent output across varied cases.

### 4.7.3 Error Analysis


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Put all predictions in a DataFrame for comparison
errors_df = pd.DataFrame({
    "y_true": y_val,
    "Text_LR": pred_text_lr,
    "Meta_GB": pred_meta_gb,
    "Text_RF": pred_text_rf,
    "Meta_RF": pred_meta_rf,
    "Text_SVM": pred_text_svm,
    "Meta_SVM": pred_meta_svm,
    "Majority": maj_vote
})

# Add columns to flag misclassifications for each model
for col in errors_df.columns[1:]:
    errors_df[col + "_err"] = errors_df[col] != errors_df["y_true"]

# Count total errors for each model
error_counts = errors_df.filter(like="_err").sum().sort_values()
print("Error counts:\n", error_counts)

# Confusion matrices
print("\nConfusion matrix for Majority Vote:")
print(confusion_matrix(y_val, errors_df["Majority"]))
print("\nClassification report for Majority Vote:")
print(classification_report(y_val, errors_df["Majority"], zero_division=0))

# Where do models fail
all_fail_mask = errors_df[[c for c in errors_df.columns if c.endswith("_err")]].all(axis=1)
all_fail_cases = errors_df[all_fail_mask]
print(f"\nCases where all models fail: {len(all_fail_cases)}")

# Where ensemble succeeds / specialists fail
specialist_fail = (errors_df["Text_LR_err"] & errors_df["Meta_GB_err"])
ensemble_success = ~errors_df["Majority_err"]
rescued_cases = errors_df[specialist_fail & ensemble_success]
print(f"Cases rescued by ensemble: {len(rescued_cases)}")




disagreements = errors_df[(errors_df["Text_LR"] != errors_df["Meta_GB"]) |
                          (errors_df["Text_LR"] != errors_df["Majority"]) |
                          (errors_df["Meta_GB"] != errors_df["Majority"])]
print("\nSample disagreements:")
print(disagreements.head(10))


The majority vote model achieved an accuracy of 78 percent with a precision of 0.85 for the positive class and 0.73 for the negative class. It was better at correctly identifying negatives, with an 87 percent recall for class 0, while its recall for class 1 was lower at 70 percent. There were 66 cases where every single model made the wrong prediction, which suggests certain examples are inherently difficult for all approaches. Interestingly, the ensemble did not rescue any cases that both specialized models got wrong, meaning most of its gains came from breaking ties or aligning with the stronger specialist. Looking at disagreements, many involved one or more redundant models voting differently from the specialists, which occasionally pushed the ensemble toward the correct answer but sometimes led it astray. Overall, the majority vote provided balanced performance, but the toughest errors remain unsolved and may require additional feature engineering or a different modeling approach.

## 4.8 Results and Discussion


i) the hybrid setup combined a text-focused model with a metadata-focused model, each handling the parts of the dataset they were best at. The text model used TF-IDF features from over 47,000 terms and leaned on logistic regression to pick up on patterns in word usage. This paid off especially for terms like “recipe,” “baking,” and “ingredients,” which were strongly tied to evergreen content. The metadata model worked with 25 numerical features, and tree-based methods were used to make sense of things like link ratios and non-markup character counts. By splitting the workload like this, the models could each specialize, and their predictions were later blended into a final result that took advantage of both perspectives.

ii) The redundant modeling approach really helped balance things out when one model struggled. For example, the text model was great at classifying pages rich in relevant keywords, but when the page had sparse or misleading text, the metadata model could still catch signals from structural features. Combining them kept performance consistent across different content types, which is clear from the boost in F1 scores when the results were merged. This setup meant fewer extreme misclassifications, especially in tricky cases where a single model might have been overconfident for the wrong reasons. It gave the overall system a safety net.

iii) One of the bigger challenges was the heavy class imbalance in the dataset, since evergreen pages were less common. Without adjustments, models tended to lean toward predicting the more frequent non-evergreen label, which hurt recall. Another hurdle was making sure the large TF-IDF feature space didn’t cause overfitting, especially with terms that looked predictive in training but weren’t in testing. Keeping model complexity in check and using cross-validation helped with that. There was also some work involved in handling missing values in key metadata fields, which was addressed with consistent imputation so that no model got tripped up by gaps in the data.

iv) Using multiple models made the system more flexible and resilient. Some web pages in the dataset had rich, descriptive text, while others had sparse text but revealing metadata, and the hybrid approach could handle both types well. This kind of setup is great for real-world scenarios where inputs can vary a lot, like classifying news articles, detecting spam, or sorting user-generated content. On the flip side, managing multiple models takes more computing power and careful coordination, and the training pipeline is more complex than a single-model solution. Even so, the payoff was worth it because the blended predictions consistently outperformed the individual ones.

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 5. Explainable AI with LIME


## 5.1 Load the Heart Disease Dataset


In [ ]:
heart_path = (
    DATA_DIR
    / "heart_disease"
    / "HeartDiseaseTrain-Test.csv"
)

if not heart_path.is_file():
    raise FileNotFoundError(
        "The heart-disease dataset was not found.\n\n"
        f"Expected location:\n{heart_path}"
    )

df = pd.read_csv(heart_path)

df.head()

## 5.2 Preprocess Features and Create Train-Test Splits


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df_clean = df.copy()

label_encoders = {}
for col in df_clean.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le

X = df_clean.drop('target', axis=1)
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 5.3 Train a Random Forest Classifier


In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)


## 5.4 Evaluate Classification Performance


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


## 5.5 Explain Individual Predictions with LIME


In [ ]:
import lime
import lime.lime_tabular
import numpy as np

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=X.columns.tolist(),
    class_names=['No Disease', 'Disease'],
    mode='classification'
)

i = 5  # Index of test instance to explain
exp = explainer.explain_instance(X_test_scaled[i], clf.predict_proba, num_features=10)
exp.show_in_notebook(show_all=False)


## 5.6 Interpretability, Trustworthiness, and Limitations


i) The Random Forest model performed really well with strong accuracy and perfect precision. But in healthcare, it is not enough for a model to be accurate, it also needs to be easy to understand. That is where LIME helps. It shows which features influenced a specific prediction, making the decision process much clearer. This kind of explanation builds trust, especially when the predictions could affect real medical decisions.

ii) The LIME explanation lined up well with the model’s prediction of heart disease. Features like chest pain type, oldpeak, and age played major roles, which matches what we know medically. LIME clearly showed how each feature pushed the prediction toward either Disease or No Disease. This helped confirm that the model is using meaningful and relevant information. It also made the prediction feel more reliable and easier to understand.

iii) There were a few small challenges at the beginning. Some of the columns were categorical, so they needed to be encoded before the model could use them. LIME was not installed at first, so that had to be added manually. After getting through those steps and doing the right preprocessing, everything worked as expected. It just took a bit of setup at the start.

iv) LIME made it easy to see which features had the most influence on specific predictions. Features like chest pain type, thalassemia, and the number of vessels seen during fluoroscopy were especially important. The explanation also showed whether each feature increased or decreased the chance of disease. That kind of insight helps explain how the model makes decisions. It also connects nicely with real-world medical reasoning.

v) There are a few ways this project could be taken further. Trying different models like XGBoost or neural networks might give even better results. It could also help to create new features that combine medical indicators in smarter ways. Using another tool like SHAP could give more detailed explanations alongside LIME. Finally, turning this into an interactive app for clinicians or analysts would make it more useful in practice, especially if we also check for fairness and possible bias in the predictions.